<a href="https://colab.research.google.com/github/ReposofPriyanka/flyrank-ai-internship-ml-track/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review



## 1. My rule and its reason codes

### My baseline rule

I will prioritize pages for a CTR review when they have meaningful search visibility, rank between positions 4 and 20, and have a CTR below the typical CTR for their position band.

The score estimates the potential clicks associated with the observed CTR gap. It is a prioritization score, not a prediction of future traffic or proof that changing a page will improve performance.

### Reason code

`HIGH_VOLUME_LOW_CTR_FOR_POSITION`

### Action

`REVIEW_CTR` — send the page for human review of its title, snippet, and search intent.

Pages that do not meet the rule receive `NO_ACTION` in this baseline. This does not mean that those pages never need review.

In [1]:
# Setup, March 2026 page-level data, and two signal checks

!pip -q install duckdb pandas pyarrow

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from pathlib import Path

# -------------------------
# 1. Connect to FlyRank data
# -------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is missing. Add your Hugging Face READ token "
        "in Colab Secrets and enable notebook access."
    )

con = duckdb.connect()

# Escape quotes in the token before creating the secret.
safe_token = HF_TOKEN.replace("'", "''")

con.execute(
    f"CREATE SECRET (TYPE huggingface, TOKEN '{safe_token}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

# -------------------------
# 2. Load the March 2026 slice
# -------------------------

# One row per client-content pair after aggregation.
# Only March 2026 data is used.
# GSC availability must explicitly be TRUE.

query = f"""
WITH march_daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        gsc_avg_position
    FROM read_parquet(
        '{REL}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
),
page_month AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_sum_position) AS sum_position,
        COUNT(*) AS observed_days
    FROM march_daily
    GROUP BY
        client_hash_id,
        content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    impressions,
    clicks,
    sum_position,
    observed_days,
    CASE
        WHEN impressions > 0
        THEN clicks * 100.0 / impressions
        ELSE NULL
    END AS ctr,
    CASE
        WHEN impressions > 0
        THEN sum_position * 1.0 / impressions
        ELSE NULL
    END AS avg_position
FROM page_month
WHERE impressions > 0
"""

df = con.sql(query).df()

if df.empty:
    raise ValueError(
        "No March 2026 rows were returned. Check warehouse access "
        "and the month partition."
    )

# Keep only rows with usable position data for this rule.
df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df = df.replace([np.inf, -np.inf], np.nan)

df = df.dropna(
    subset=["impressions", "ctr", "avg_position"]
).copy()

df = df[df["avg_position"] > 0].copy()

# Stable internal row identifier for this notebook.
df = df.reset_index(drop=True)
df["row_id"] = np.arange(1, len(df) + 1)

print("March 2026 page-level dataset")
print("Rows:", len(df))
print("Unique content items:", df["content_hash_id"].nunique())
print("Unique clients:", df["client_hash_id"].nunique())
print("Total impressions:", int(df["impressions"].sum()))

df.head()

# -------------------------
# 3. Signal check 1:
# CTR versus search position
# -------------------------

# Position groups are descriptive buckets.
# Position 4-20 is the range used by the baseline rule.

position_bins = [0, 3, 10, 20, 50, np.inf]

position_labels = [
    "1-3",
    "4-10",
    "11-20",
    "21-50",
    "51+"
]

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

# Print n for every bucket.
position_check = (
    df.groupby(
        "position_bucket",
        observed=False
    )
    .agg(
        n=("row_id", "size"),
        median_ctr=("ctr", "median"),
        median_impressions=("impressions", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — CTR versus position")
display(position_check)

# -------------------------
# 4. Signal check 2:
# Impression volume
# -------------------------

# These are fixed, readable volume buckets.
# They are not labels and are not fitted from outcomes.

volume_bins = [0, 100, 500, 2000, np.inf]

volume_labels = [
    "1-99",
    "100-499",
    "500-1999",
    "2000+"
]

df["volume_bucket"] = pd.cut(
    df["impressions"],
    bins=volume_bins,
    labels=volume_labels,
    include_lowest=True,
    right=False
)

volume_check = (
    df.groupby(
        "volume_bucket",
        observed=False
    )
    .agg(
        n=("row_id", "size"),
        median_impressions=("impressions", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

print("SIGNAL 2 — Impression volume")
display(volume_check)

# -------------------------
# 5. Print evidence for verdicts
# -------------------------

print("Signal 1 evidence:")
print(
    "Position 4-10 median CTR:",
    df.loc[df["position_bucket"] == "4-10", "ctr"].median()
)
print(
    "Position 11-20 median CTR:",
    df.loc[df["position_bucket"] == "11-20", "ctr"].median()
)
print(
    "Position 21-50 median CTR:",
    df.loc[df["position_bucket"] == "21-50", "ctr"].median()
)

print("\nSignal 2 evidence:")
print(
    "Median impressions:",
    df["impressions"].median()
)
print(
    "75th percentile impressions:",
    df["impressions"].quantile(0.75)
)

print(
    "\nReview the displayed bucket tables before writing "
    "your final signal verdicts."
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March 2026 page-level dataset
Rows: 175304
Unique content items: 175304
Unique clients: 47
Total impressions: 280655033
SIGNAL 1 — CTR versus position


,position_bucket,n,median_ctr,median_impressions
0,1-3,17426,0.0,316.5
1,4-10,83288,0.0,204.0
2,11-20,29922,0.0,238.0
3,21-50,32240,0.0,180.0
4,51+,12428,0.0,52.0


SIGNAL 2 — Impression volume


,volume_bucket,n,median_impressions,median_ctr
0,1-99,73863,13.0,0.000000
1,100-499,39517,226.0,0.000000
2,500-1999,32047,962.0,0.155039
3,2000+,29877,4724.0,0.206954


Signal 1 evidence:
Position 4-10 median CTR: 0.0
Position 11-20 median CTR: 0.0
Position 21-50 median CTR: 0.0

Signal 2 evidence:
Median impressions: 178.0
75th percentile impressions: 1053.0

Review the displayed bucket tables before writing your final signal verdicts.


In [2]:
# Verify position bucket counts and display CTR precisely

print("Dataset rows:", len(df))

print(
    "Position bucket count total:",
    position_check["n"].sum()
)

print("\nPosition bucket table — precise CTR values")

position_check["median_ctr"] = (
    position_check["median_ctr"].map(
        lambda x: f"{x:.6f}%"
    )
)

display(position_check)

print("\nPosition bucket counts from raw dataframe")

display(
    df["position_bucket"]
    .value_counts(dropna=False)
    .sort_index()
)

Dataset rows: 175304
Position bucket count total: 175304

Position bucket table — precise CTR values


,position_bucket,n,median_ctr,median_impressions
0,1-3,17426,0.000000%,316.5
1,4-10,83288,0.000000%,204.0
2,11-20,29922,0.000000%,238.0
3,21-50,32240,0.000000%,180.0
4,51+,12428,0.000000%,52.0



Position bucket counts from raw dataframe


,count
position_bucket,
1-3,17426
4-10,83288
11-20,29922
21-50,32240
51+,12428


In [3]:
# ML-07 — Position signal diagnostic
# Compare median CTR with impression-weighted CTR.

position_weighted_check = (
    df.groupby("position_bucket", observed=False)
    .agg(
        n=("row_id", "size"),
        total_clicks=("clicks", "sum"),
        total_impressions=("impressions", "sum"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

position_weighted_check["weighted_ctr"] = (
    position_weighted_check["total_clicks"]
    / position_weighted_check["total_impressions"]
    * 100
)

position_weighted_check["median_ctr"] = (
    position_weighted_check["median_ctr"].round(6)
)

position_weighted_check["weighted_ctr"] = (
    position_weighted_check["weighted_ctr"].round(6)
)

display(position_weighted_check)

print(
    "Pages with zero CTR:",
    int((df["ctr"] == 0).sum())
)

print(
    "Pages with non-zero CTR:",
    int((df["ctr"] > 0).sum())
)

,position_bucket,n,total_clicks,total_impressions,median_ctr,weighted_ctr
0,1-3,17426,160503.0,41417584.0,0.0,0.387524
1,4-10,83288,481189.0,148112436.0,0.0,0.324881
2,11-20,29922,98488.0,31191659.0,0.0,0.315751
3,21-50,32240,80635.0,57614383.0,0.0,0.139956
4,51+,12428,958.0,2318971.0,0.0,0.041311


Pages with zero CTR: 106524
Pages with non-zero CTR: 68780


In [4]:
# Final row-count and aggregation consistency check

print("Current dataframe rows:", len(df))
print("Unique row IDs:", df["row_id"].nunique())
print("Total impressions:", df["impressions"].sum())
print("Total clicks:", df["clicks"].sum())

print("\nPosition bucket counts:")
print(
    df["position_bucket"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nPosition table counts total:")
print(position_weighted_check["n"].sum())

print("\nPosition table impressions total:")
print(position_weighted_check["total_impressions"].sum())

print("\nCurrent dataframe impressions total:")
print(df["impressions"].sum())

Current dataframe rows: 175304
Unique row IDs: 175304
Total impressions: 280655033.0
Total clicks: 821773.0

Position bucket counts:
position_bucket
1-3      17426
4-10     83288
11-20    29922
21-50    32240
51+      12428
Name: count, dtype: int64

Position table counts total:
175304

Position table impressions total:
280655033.0

Current dataframe impressions total:
280655033.0


## 2. Build the ranked queue (writes the CSV)

### Signal verdicts

1. **CTR versus position — CONFIRMED**  
   The bucket table shows that impression-weighted CTR decreases from 0.387524% for positions 1–3 to 0.041311% for positions 51+. This supports the assumption because pages with better search positions generally have higher CTR in the observed data. However, the median CTR is 0.0% across all position buckets, so the weighted CTR provides a more informative comparison.

2. **Impression volume — CONFIRMED**  
   The bucket table shows that median CTR increases from 0.00000% in the 1–99 and 100–499 impression buckets to 0.15509% in the 500–1,999 bucket and 0.20694% in the 2,000+ bucket. This supports the assumption because higher-volume pages have higher median CTR in the observed data. However, this is an association and does not prove that impression volume causes higher CTR.

In [5]:
# Build one transparent baseline rule and write the ranked queue.

import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------
# 1. Prepare numeric columns
# -------------------------

df["impressions"] = pd.to_numeric(
    df["impressions"], errors="coerce"
)

df["clicks"] = pd.to_numeric(
    df["clicks"], errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

df["avg_position"] = pd.to_numeric(
    df["avg_position"], errors="coerce"
)

# Remove rows with missing values required by the rule.
df = df.dropna(
    subset=[
        "impressions",
        "clicks",
        "ctr",
        "avg_position",
        "position_bucket"
    ]
).copy()

# -------------------------
# 2. Calculate position CTR benchmarks
# -------------------------

# Calculate impression-weighted CTR for each position bucket.

position_ctr_benchmark = (
    df.groupby("position_bucket", observed=True)
    .agg(
        total_clicks=("clicks", "sum"),
        total_impressions=("impressions", "sum")
    )
)

position_ctr_benchmark["benchmark_ctr"] = (
    position_ctr_benchmark["total_clicks"]
    / position_ctr_benchmark["total_impressions"]
    * 100
)

# Convert the benchmark table into a normal dictionary.
# This avoids categorical mapping issues.

benchmark_lookup = (
    position_ctr_benchmark["benchmark_ctr"]
    .to_dict()
)

# Convert categorical bucket labels to strings before mapping.
df["position_ctr_benchmark"] = (
    df["position_bucket"]
    .astype(str)
    .map(benchmark_lookup)
)

df["position_ctr_benchmark"] = pd.to_numeric(
    df["position_ctr_benchmark"],
    errors="coerce"
)

print("Position CTR benchmarks:")
display(position_ctr_benchmark)

print(
    "Pages with a benchmark:",
    df["position_ctr_benchmark"].notna().sum()
)

# -------------------------
# 3. Define the flags
# -------------------------

# High volume = at or above the 75th percentile.
volume_threshold = df["impressions"].quantile(0.75)

df["high_volume"] = (
    df["impressions"] >= volume_threshold
)

# Position 4-20.
df["target_position"] = (
    df["position_bucket"]
    .astype(str)
    .isin(["4-10", "11-20"])
)

# CTR below the impression-weighted benchmark.
df["below_position_ctr"] = (
    df["ctr"] < df["position_ctr_benchmark"]
)

# Only pages meeting all three conditions are flagged.
df["flag_for_review"] = (
    df["high_volume"]
    & df["target_position"]
    & df["below_position_ctr"]
    & df["position_ctr_benchmark"].notna()
)

# -------------------------
# 4. Calculate score
# -------------------------

# CTR is expressed as a percentage.
# Estimated click gap = impressions × CTR gap / 100.

df["ctr_gap"] = (
    df["position_ctr_benchmark"] - df["ctr"]
).clip(lower=0)

df["score"] = np.where(
    df["flag_for_review"],
    df["impressions"] * df["ctr_gap"] / 100,
    0.0
)

# -------------------------
# 5. Assign reason code and action
# -------------------------

df["reason_code"] = np.where(
    df["flag_for_review"],
    "HIGH_VOLUME_LOW_CTR_FOR_POSITION",
    "NOT_FLAGGED"
)

df["action"] = np.where(
    df["flag_for_review"],
    "REVIEW_CTR",
    "NO_ACTION"
)

# -------------------------
# 6. Rank the queue
# -------------------------

ranked = df.sort_values(
    by=["score", "impressions"],
    ascending=[False, False]
).copy()

ranked["rank"] = np.arange(1, len(ranked) + 1)

# -------------------------
# 7. Select output columns
# -------------------------

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "position_bucket",
    "position_ctr_benchmark",
    "ctr_gap",
    "score",
    "reason_code",
    "action"
]

baseline_queue = ranked[queue_columns].copy()

# -------------------------
# 8. Write the CSV
# -------------------------

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

baseline_queue.to_csv(output_path, index=False)

print("\nBaseline queue created.")
print("Rows:", len(baseline_queue))
print(
    "Flagged for review:",
    int(df["flag_for_review"].sum())
)
print("Volume threshold:", round(volume_threshold, 2))
print("Saved to:", output_path)

display(baseline_queue.head(10))

Position CTR benchmarks:


,total_clicks,total_impressions,benchmark_ctr
position_bucket,,,
1-3,160503.0,41417584.0,0.387524
4-10,481189.0,148112436.0,0.324881
11-20,98488.0,31191659.0,0.315751
21-50,80635.0,57614383.0,0.139956
51+,958.0,2318971.0,0.041311


Pages with a benchmark: 175304

Baseline queue created.
Rows: 175304
Flagged for review: 19155
Volume threshold: 1053.0
Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,impressions,clicks,ctr,avg_position,position_bucket,position_ctr_benchmark,ctr_gap,score,reason_code,action
7624,1,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,0.030066,3.166132,4-10,0.324881,0.294815,421.641400,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
40641,2,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,0.062598,5.948459,4-10,0.324881,0.262283,347.769318,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
117674,3,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,0.013943,9.735658,4-10,0.324881,0.310938,334.519856,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
40670,4,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,0.153389,3.396293,4-10,0.324881,0.171492,292.922550,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
108214,5,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,0.004478,7.831807,4-10,0.324881,0.320403,286.222596,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
128479,6,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,0.185760,4.551516,4-10,0.324881,0.139121,270.363775,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
155029,7,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,0.007161,7.208276,4-10,0.324881,0.317720,266.211200,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
132052,8,client_73cda7b4e4f265ea,content_f43118e089ecc69a,139417.0,191.0,0.136999,5.342218,4-10,0.324881,0.187882,261.939190,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
52273,9,client_a80fca3f171ed1de,content_9540d884af3e41fd,82376.0,11.0,0.013353,8.005184,4-10,0.324881,0.311527,256.623882,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
109166,10,client_73cda7b4e4f265ea,content_425715547c6a3ea8,71513.0,3.0,0.004195,6.983444,4-10,0.324881,0.320686,229.332071,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR


## 3. Top-20 review


In [6]:
# Top-20 review

top20 = baseline_queue.head(20).copy()

# Create a readable explanation for each row.
top20["why_selected"] = top20.apply(
    lambda row: (
        f"{int(row['impressions']):,} impressions; "
        f"position {row['avg_position']:.1f}; "
        f"CTR {row['ctr']:.4f}% versus "
        f"{row['position_ctr_benchmark']:.4f}% "
        "position-band benchmark."
        if row["action"] == "REVIEW_CTR"
        else "Not selected by the baseline rule; score is zero."
    ),
    axis=1
)

# Draft a limitation for each row.
top20["what_would_make_it_wrong"] = top20.apply(
    lambda row: (
        "The benchmark may not fit this page's query intent; "
        "SERP features may affect CTR; or the page may already "
        "satisfy users despite its lower CTR."
        if row["action"] == "REVIEW_CTR"
        else (
            "The page may still need review for reasons not "
            "captured by this rule, such as stale content "
            "or a technical issue."
        )
    ),
    axis=1
)

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "score",
        "why_selected",
        "what_would_make_it_wrong"
    ]
].copy()

display(top20_review)

# Save a review draft.
review_path = output_dir / "top20_review.csv"
top20_review.to_csv(review_path, index=False)

print("Top-20 review rows:", len(top20_review))
print("Saved to:", review_path)

,rank,content_hash_id,action,reason_code,score,why_selected,what_would_make_it_wrong
7624,1,content_34a70fea29d15f24,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,421.641400,"143,019 impressions; position 3.2; CTR 0.0301%...",The benchmark may not fit this page's query in...
40641,2,content_7c6373141eae744a,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,347.769318,"132,593 impressions; position 5.9; CTR 0.0626%...",The benchmark may not fit this page's query in...
117674,3,content_f6116743b00afc2d,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,334.519856,"107,584 impressions; position 9.7; CTR 0.0139%...",The benchmark may not fit this page's query in...
40670,4,content_acbcc847f8996314,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,292.922550,"170,808 impressions; position 3.4; CTR 0.1534%...",The benchmark may not fit this page's query in...
108214,5,content_cd3d932d4e1c8db0,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,286.222596,"89,332 impressions; position 7.8; CTR 0.0045% ...",The benchmark may not fit this page's query in...
128479,6,content_b99ea6861864dea5,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,270.363775,"194,337 impressions; position 4.6; CTR 0.1858%...",The benchmark may not fit this page's query in...
155029,7,content_046fc480045b88f5,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,266.211200,"83,788 impressions; position 7.2; CTR 0.0072% ...",The benchmark may not fit this page's query in...
132052,8,content_f43118e089ecc69a,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,261.939190,"139,417 impressions; position 5.3; CTR 0.1370%...",The benchmark may not fit this page's query in...
52273,9,content_9540d884af3e41fd,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,256.623882,"82,376 impressions; position 8.0; CTR 0.0134% ...",The benchmark may not fit this page's query in...
109166,10,content_425715547c6a3ea8,REVIEW_CTR,HIGH_VOLUME_LOW_CTR_FOR_POSITION,229.332071,"71,513 impressions; position 7.0; CTR 0.0042% ...",The benchmark may not fit this page's query in...


Top-20 review rows: 20
Saved to: work/outputs/top20_review.csv


## 4. Weak picks + leakage check



In [7]:
# Weak picks and final self-check

# -------------------------
# 1. Inspect weak picks
# -------------------------

# Lowest-scoring pages that were still flagged.
weak_picks = (
    baseline_queue[
        baseline_queue["action"] == "REVIEW_CTR"
    ]
    .sort_values("score", ascending=True)
    .head(5)
)

print("Weak picks — lowest-scoring flagged pages")

display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "impressions",
            "ctr",
            "avg_position",
            "position_ctr_benchmark",
            "score",
            "reason_code",
            "action"
        ]
    ]
)

# -------------------------
# 2. Leakage checks
# -------------------------

forbidden_columns = [
    "future_clicks",
    "future_impressions",
    "future_ctr",
    "future_position",
    "is_declining_label",
    "trend_pct",
    "trend_direction"
]

present_forbidden = [
    col for col in forbidden_columns
    if col in df.columns
]

assert not present_forbidden, (
    f"Potential leakage columns found: {present_forbidden}"
)

# Required output columns.
required_columns = [
    "rank",
    "score",
    "reason_code",
    "action"
]

assert all(
    col in baseline_queue.columns
    for col in required_columns
)

assert baseline_queue["rank"].is_monotonic_increasing
assert baseline_queue["score"].notna().all()

# Confirm the CSV exists.
assert output_path.exists()

# Confirm the top-20 review contains 20 rows.
assert len(top20_review) == 20

print("\nLeakage check passed.")
print("Required output columns passed.")
print("CSV exists:", output_path.exists())

# -------------------------
# 3. Final summary
# -------------------------

print("\nML-07 Summary")
print("Pages scored:", len(baseline_queue))
print(
    "Pages flagged:",
    int((baseline_queue["action"] == "REVIEW_CTR").sum())
)
print("Top-20 rows reviewed:", len(top20_review))
print("Weak picks inspected:", len(weak_picks))
print("CSV:", output_path)


Weak picks — lowest-scoring flagged pages


,rank,content_hash_id,impressions,ctr,avg_position,position_ctr_benchmark,score,reason_code,action
94533,19155,content_16677704942bd5e0,1584.0,0.315657,15.679293,0.315751,0.001497,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
118764,19154,content_1aae0c3a544a6f07,1232.0,0.324675,8.181818,0.324881,0.002533,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
21262,19153,content_ed474153afa3ec42,1232.0,0.324675,3.586039,0.324881,0.002533,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
119969,19152,content_1f762a1ae4aedc89,1585.0,0.315457,13.408202,0.315751,0.004655,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR
29362,19151,content_4763692f479644ed,7697.0,0.324802,5.640899,0.324881,0.006082,HIGH_VOLUME_LOW_CTR_FOR_POSITION,REVIEW_CTR



Leakage check passed.
Required output columns passed.
CSV exists: True

ML-07 Summary
Pages scored: 175304
Pages flagged: 19155
Top-20 rows reviewed: 20
Weak picks inspected: 5
CSV: work/outputs/baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.